# Reflection Extraction Notebook

This notebook recursively finds all reflection experiments in a root directory and extracts the self-reflections directly from the `llm_traces` logs.

A directory is considered a reflection experiment if it contains a `reflections` folder with a `reflections.json` file. The data is pulled from the `llm_traces` subdirectory.

In [3]:
import json
import os
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm

def extract_reflections_from_traces(root_dir):
    results = []
    root_path = Path(root_dir)
    
    # We first find experiments that qualify as reflection experiments
    # Requirement: must have a reflections/system_id/reflections.json file
    qualifying_exps = set()
    for refl_json in root_path.rglob("reflections.json"):
        # exp_dir is 3 levels up from reflections/system_id/reflections.json
        qualifying_exps.add(refl_json.parent.parent.parent)
    
    print(f"Found {len(qualifying_exps)} qualifying reflection experiments.")
    
    for exp_dir in tqdm(qualifying_exps):
        trace_dir = exp_dir / "llm_traces"
        if not trace_dir.exists():
            continue
            
        # Find all trace files recursively in llm_traces
        trace_files = list(trace_dir.rglob("episode_*.jsonl"))
        
        for trace_path in trace_files:
            # Path structure: .../llm_traces/system_id/episode_NNN.jsonl
            system_id = trace_path.parent.name
            episode_id = trace_path.stem.replace("episode_", "")
            
            with open(trace_path, 'r') as f:
                for line in f:
                    try:
                        data = json.loads(line)
                        if data.get("component") == "self_reflection":
                            reflection = data.get("output", {}).get("reflection")
                            if reflection:
                                results.append({
                                    "exp_name": exp_dir.name,
                                    "system_id": system_id,
                                    "episode_id": episode_id,
                                    "model": data.get("model", "unknown"),
                                    "reflection": reflection,
                                    "trace_path": str(trace_path)
                                })
                    except json.JSONDecodeError:
                        continue
            
    return pd.DataFrame(results)

# Set your root results directory here
ROOT_RESULTS = "../results"
df_reflections = extract_reflections_from_traces(ROOT_RESULTS)

if not df_reflections.empty:
    # Sort by exp, system, and episode for readability
    df_reflections = df_reflections.sort_values(["exp_name", "system_id", "episode_id"])
    
df_reflections.head()

Found 25 qualifying reflection experiments.


100%|██████████| 25/25 [00:24<00:00,  1.00it/s]


,exp_name,system_id,episode_id,model,reflection,trace_path
353,llm_react_orchestrator_systems_quaternary_n10_...,Au-Cr-Cs-Dy,000,openai/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8,"- Prioritize Au-Dy and Au-Cs binary systems, e...",../results/smoke/custom/qwen3-30b-instr-reflx/...
952,llm_react_orchestrator_systems_quaternary_n10_...,Au-Cr-Cs-Dy,000,openai/Qwen/Qwen3.5-122B-A10B-FP8,* Prioritize Au-Cs and Au-Dy binary stoichio...,../results/smoke/custom/qwen3dot5-122b-reflx/s...
348,llm_react_orchestrator_systems_quaternary_n10_...,Au-Cr-Cs-Dy,001,openai/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8,- Prioritize Au1Cs1 with 20–25 queries in the ...,../results/smoke/custom/qwen3-30b-instr-reflx/...
949,llm_react_orchestrator_systems_quaternary_n10_...,Au-Cr-Cs-Dy,001,openai/Qwen/Qwen3.5-122B-A10B-FP8,* **Prioritize Au-Cs binaries:** Shift 60%+ ...,../results/smoke/custom/qwen3dot5-122b-reflx/s...
352,llm_react_orchestrator_systems_quaternary_n10_...,Au-Cr-Cs-Dy,002,openai/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8,- Prioritize Au1Cs1 with 20–25 queries in the ...,../results/smoke/custom/qwen3-30b-instr-reflx/...


## Summary Statistics

In [7]:
if not df_reflections.empty:
    print(f"Total reflections extracted: {len(df_reflections)}")
    print(f"Unique experiments: {df_reflections['exp_name'].nunique()}")
    print(f"Unique systems: {df_reflections['system_id'].nunique()}")
    print(f"Models: {df_reflections['model'].unique()}")
    
    # Display reflections for a sample system
    sample_row = df_reflections.iloc[3]
    sample_system = sample_row['system_id']
    sample_exp = sample_row['exp_name']
    
    print(f"\nSample reflections for {sample_system} in {sample_exp}:")
    system_df = df_reflections[(df_reflections['system_id'] == sample_system) & 
                               (df_reflections['exp_name'] == sample_exp)]
    
    for _, row in system_df.iterrows():
        print(f"--- Episode {row['episode_id']} ---")
        print(row['reflection'])
        print()

Total reflections extracted: 1040
Unique experiments: 8
Unique systems: 31
Models: ['openai/Qwen/Qwen3-30B-A3B-Instruct-2507-FP8'
 'openai/Qwen/Qwen3.5-122B-A10B-FP8']

Sample reflections for Au-Cr-Cs-Dy in llm_react_orchestrator_systems_quaternary_n10_maxatoms20_intermetallic_smact_10systems_40queries_100stabilitymeV:
--- Episode 000 ---
- Prioritize Au-Dy and Au-Cs binary systems, especially Au1Cs1 and Au3Dy1, which were highly productive in generating stable and novel structures.
- Avoid or drastically reduce queries into Au1Dy2 and Cr-containing compositions (Au1Cr1, Cr1Cs1, Cr1Dy1), which yielded no stable structures despite novelty.
- Allocate more queries (e.g., 20–25 total) to Au3Dy1 and Au1Cs1 in the next episode, given their high yield of stable discoveries.
- Use early stable discoveries (e.g., Au1Cs1 at e=0.000) as signals to increase exploration in that compositional region.
- Limit exploration of high-energy, unstable regions (e.g., Au1Dy2) after initial probing to avoid 